# Conditional cubes and conditional observables

A **conditional cube** (`ConditionalLeafCubeKind`) takes one of two cube kinds at
runtime, chosen by the parity of a correlation surface measured earlier: its
`condition`. This is the classical feedforward of gate teleportation, e.g. the
S correction after a T injection.

`tqec` compiles such a computation once, to Stim text with `IF`/`ELSE` blocks.
Stim cannot run that text, so the same graph also gives the ordinary
`stim.Circuit` of any single branch, noisy or not, for sampling and decoding.

This notebook builds a *branching tree* of conditional cubes, compiles it, takes
its branches apart, and then adds a **conditional observable**: a logical
observable whose shape depends on a conditional cube's branch.

In [ ]:
from tqec.computation.block_graph import BlockGraph
from tqec.computation.correlation import (
    ConditionalCorrelationSurface,
    CorrelationSurface,
    ZXEdge,
    ZXNode,
)
from tqec.computation.cube import ConditionalLeafCubeKind
from tqec.utils.enums import Basis
from tqec.utils.position import Position3D, Direction3D
from tqec import NoiseModel, compile_block_graph

## A branching tree

A column of memory cubes where, at every layer `z`, a lattice-surgery merge with a
side cube conditions a cube at `z + 1`: each condition is the outcome of that
merge.

In [ ]:
def build_branching_tree(n: int) -> BlockGraph:
    """Build a block graph with ``n`` conditional cubes.

    An ``n = 2`` tree (M: main column, S: side cube, C: conditional cube)::

           M  C    z = 3
           |  |
        C  M--S    z = 2
        |  |
        S--M       z = 1
           |
           M       z = 0

    Each conditional cube is conditioned on the merge (the S--M pipe) on the
    layer below it.
    """
    if n < 1:
        raise ValueError(f"The number of conditional cubes must be at least 1. Got {n} instead.")

    g = BlockGraph(f"tree_{n}")

    def add_branch_at_z(g: BlockGraph, z: int):
        # sign parity
        s = 1 - 2 * (z % 2)

        p = Position3D(0, 0, z)
        b = Position3D(s, 0, z)
        c = Position3D(s, 0, z + 1)

        condition = CorrelationSurface(
            span=frozenset({ZXEdge(ZXNode(p, Basis.X), ZXNode(b, Basis.X))})
        )

        g.add_cube(p, "ZXZ")
        g.add_cube(b, "ZXZ")
        g.add_cube(
            c,
            ConditionalLeafCubeKind.ZXZ_ZXX,
            condition=condition
        )

        g.add_pipe(p, b)
        g.add_pipe(b, c)

        return p

    bottom = Position3D(0, 0, 0)
    top = Position3D(0, 0, n + 1)
    branch_positions = [add_branch_at_z(g, z) for z in range(1, n + 1)]

    g.add_cube(bottom, "ZXZ")
    g.add_cube(top, "ZXZ")

    g.add_pipe(bottom, branch_positions[0])
    for p1, p2 in zip(branch_positions[:-1], branch_positions[1:]):
        g.add_pipe(p1, p2)
    g.add_pipe(branch_positions[-1], top)


    return g

In [ ]:
g = build_branching_tree(3)
g.view_as_html()

## The `IF`/`ELSE` text

`generate_stim_text` emits one circuit for every branch at once: the instructions
that differ between a conditional cube's two kinds sit in `IF`/`ELSE` blocks,
conditioned on the parity of the measurements its condition reads.

Compiling a conditional cube warns that the correlation-surface validity check is
skipped: PyZX cannot represent a conditional cube, so its condition is not
re-checked against the graph. This is expected.

In [ ]:
def make_stim_text(g: BlockGraph, k: int = 1) -> str:
    return compile_block_graph(g, observables=None).generate_stim_text(k=k)


stim_text = make_stim_text(g)
first_if = stim_text.index("IF(")
print(stim_text[first_if : stim_text.index("}", stim_text.index("} ELSE {", first_if) + 1) + 1])

## One branch as a stim circuit

`generate_branch_circuit` keeps one arm of every `IF` block, named by the
conditional cube's position and its branch (`0` for the first kind of its pair,
`1` for the second). The result is an ordinary `stim.Circuit`: every detector is
deterministic, so its detector error model builds. `generate_branch_circuits`
compiles once for several branches.

In [ ]:
g2 = build_branching_tree(2)
compiled2 = compile_block_graph(g2, observables=None)
cubes = sorted((c.position for c in g2.cubes if c.is_conditional), key=lambda p: p.z)

all_bits = [(0, 0), (1, 0), (0, 1), (1, 1)]
circuits = compiled2.generate_branch_circuits(1, [dict(zip(cubes, bits)) for bits in all_bits])
for bits, circuit in zip(all_bits, circuits):
    circuit.detector_error_model()  # raises if a detector is not deterministic
    print(f"branches {bits}: {circuit.num_detectors} detectors, {circuit.num_measurements} measurements")

## Noise

Both outputs take a noise model. In the `IF`/`ELSE` text each arm is noised in
place, so every branch carries the noise it would on its own; a branch circuit is
noised like any `stim.Circuit`. A logical error rate of the whole computation is
an average over its branches.

In [ ]:
noise = NoiseModel.uniform_depolarizing(1e-3)
noisy_text = compiled2.generate_stim_text(1, noise_model=noise)
print(noisy_text[noisy_text.index("IF(") :][:600])

## A conditional observable

A logical observable that passes through a conditional cube reads it in a basis
that depends on its branch, so it has one shape per branch: a
`ConditionalCorrelationSurface`, made of the conditions it depends on and the
surface to use for each of their outcomes.

Below, a `ZXX` column is merged with a side column ending in a conditional
`ZXX_ZXZ` cube. The condition reads a lone cube placed beside the graph. With the
condition at `0` the observable runs into the conditional cube; at `1` it stays in
the main column.

The condition reads a lone cube initialised and measured in `Z`, so without noise
it is always `0`: it stands in for a real random outcome, such as the merge of a
T injection.

In [ ]:
b1, c1, c2, t1, t2 = (Position3D(0, 0, 0), Position3D(0, 0, 1), Position3D(1, 0, 1),
                      Position3D(0, 0, 2), Position3D(1, 0, 2))
source = Position3D(2, 0, 0)
condition = CorrelationSurface(span=frozenset([ZXEdge(ZXNode(source, Basis.Z), ZXNode(source, Basis.Z))]))

g_obs = BlockGraph("conditional_observable")
g_obs.add_cube(b1, "ZXX")
g_obs.add_cube(c1, "ZXZ")
g_obs.add_cube(c2, "ZXZ")
g_obs.add_cube(t1, "ZXX")
g_obs.add_cube(source, "ZXZ")
g_obs.add_cube(t2, ConditionalLeafCubeKind.ZXX_ZXZ, condition=condition)
for u, v in [(b1, c1), (c1, c2), (c1, t1), (c2, t2)]:
    g_obs.add_pipe(u, v)

x = lambda u, v: ZXEdge(ZXNode(u, Basis.X), ZXNode(v, Basis.X))
observable = ConditionalCorrelationSurface(
    conditions=(condition,),
    resolutions={
        (False,): CorrelationSurface(span=frozenset({x(b1, c1), x(c1, c2), x(c2, t2)})),
        (True,): CorrelationSurface(span=frozenset({x(b1, c1), x(c1, t1)})),
    },
)
compiled_obs = compile_block_graph(g_obs, observables=[observable])
text = compiled_obs.generate_stim_text(1)
print([line for line in text.splitlines() if "OBSERVABLE_INCLUDE" in line][:4])

The observable is emitted as a shared part plus, inside an `IF`, the
measurements that change with the branch. In each branch circuit it is an
ordinary, deterministic observable, protected to the full distance `2k + 1`.

Compiling warns that the per-branch resolutions are not checked: tqec does not yet
verify that each resolution is a valid surface of the graph with that branch
substituted in. Here both are, by construction.

In [ ]:
for branch in (0, 1):
    circuit = compiled_obs.generate_branch_circuit(2, {t2: branch}, noise_model=noise)
    distance = len(circuit.shortest_graphlike_error())
    print(f"branch {branch}: {circuit.num_observables} observable, distance {distance}")

`compile_block_graph`'s default, `observables="auto"`, only finds the observables
that touch no conditional cube, which are the same in every branch. It leaves out
the others with a warning: give them explicitly, as above.

In [ ]:
compiled_auto = compile_block_graph(g_obs)  # warns: one surface crosses the conditional cube
n_found = compiled_auto.generate_branch_circuit(1, {t2: 0}).num_observables
print(f"{n_found} observables found: one along the main column, and the lone cube")

## Scaling

The `IF`/`ELSE` text grows linearly with the number of conditional cubes (one
block group per cube), not with the number of branches, `2**n`.

In [ ]:
for n in [1, 2, 4, 8, 16, 32]:
    text = make_stim_text(build_branching_tree(n))
    print(f"n = {n:2}: {len(text.splitlines()):6} lines, {text.count('IF('):3} IF blocks")